# Step 09: Export RF-DETR Model to ONNX & TensorRT for Production Triton
Converts trained RF-DETR multi-class PyTorch model to ONNX strictly matching the production Triton `tagdet_rt` configuration:
- **Model Name**: `tagdet_rt`
- **Max Batch Size**: `1`
- **Input Binding**: `images` `[3, 640, 480]` (FP32)
- **Output Binding**: `output` `[18900, 7]` (FP32) `[cx, cy, w, h, score_0, score_1, score_2]` zero-padded
- **Engine Format**: `model.plan` (TensorRT FP16)

Includes verification via ONNXRuntime to confirm exact tensor shapes, names, and numerical ranges.


In [ ]:
# STEP 0 - Environment Dependencies (Strictly onnxruntime==1.16.1 & numpy<2.0)
# Note: numpy<2.0 prevents 'AttributeError: _ARRAY_API not found' / 'numpy.core.multiarray failed to import'
%pip install -q onnx onnxruntime==1.16.1 "numpy<2.0.0" pillow


In [ ]:
# CELL 1 - Imports, VRAM Monitor & Logger Initialization
import os, sys, json, time, math, copy, logging, random, shutil, warnings
warnings.filterwarnings("ignore", message=".*meshgrid.*")
warnings.filterwarnings("ignore", message=".*torch.cuda.amp.autocast.*")
warnings.filterwarnings("ignore", message=".*max_detection_threshold.*")
warnings.filterwarnings("ignore", category=FutureWarning)
from pathlib import Path
from typing import Dict, List, Any, Optional, Tuple
from collections import defaultdict, Counter
import urllib.parse
from urllib.parse import urlparse, unquote
from concurrent.futures import ThreadPoolExecutor, as_completed

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(line_buffering=True)
os.environ["PYTHONUNBUFFERED"] = "1"

# Configure multiprocessing sharing strategy to prevent 'Too many open files' error
import torch.multiprocessing as mp
try:
    mp.set_sharing_strategy('file_system')
except Exception:
    pass

try:
    import resource
    rlimit = resource.getrlimit(resource.RLIMIT_NOFILE)
    resource.setrlimit(resource.RLIMIT_NOFILE, (max(rlimit[0], 4096), max(rlimit[1], 4096)))
except Exception:
    pass

import torch
# Notebook 09 performs pure PyTorch -> ONNX tensor tracing with torch.randn dummy tensors.
# cv2 and supervision are optional and wrapped in try/except to prevent C-extension binary collisions.
try:
    import cv2
except Exception:
    pass

try:
    import requests
except Exception:
    pass

try:
    import supervision as sv
except Exception:
    pass
# Neutralize Matplotlib interactive callback to permanently prevent _draw_all_if_interactive crash
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
plt.ioff()
plt.close('all')
try:
    _ip = get_ipython()
    if _ip and hasattr(_ip, 'events'):
        _ip.events.callbacks['post_execute'] = [
            _cb for _cb in _ip.events.callbacks.get('post_execute', [])
            if 'draw_all' not in getattr(_cb, '__name__', '')
        ]
except Exception:
    pass
import matplotlib.patches as patches, numpy as np, pandas as pd
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF
from PIL import Image, ImageDraw, ImageFont
from dotenv import load_dotenv
from tqdm.auto import tqdm

try:
    from torchmetrics.detection.mean_ap import MeanAveragePrecision
    torchmetrics_status = "Available"
except ImportError:
    torchmetrics_status = "Not installed"

try:
    from rfdetr import RFDETRNano, RFDETRSmall, RFDETRMedium, RFDETRBase, RFDETRLarge
    from rfdetr import main as rfdetr_main
    from rfdetr.models.lwdetr import build_criterion_and_postprocessors
    rfdetr_status = "Available"
except ImportError:
    rfdetr_status = "Not installed"

# Auto-flushing console and file logger
class FlushHandler(logging.StreamHandler):
    def emit(self, record):
        super().emit(record)
        self.flush()

class FlushFileHandler(logging.FileHandler):
    def emit(self, record):
        super().emit(record)
        self.flush()

logger = logging.getLogger("rfdetr_multi_class")
logger.setLevel(logging.INFO)
logger.handlers.clear()
logger.propagate = False
log_format = "%(asctime)s | %(levelname)-8s | %(message)s"
console_handler = FlushHandler(sys.stdout)
console_handler.setFormatter(logging.Formatter(log_format))
logger.addHandler(console_handler)

def print_vram_usage(tag="Status"):
    if torch.cuda.is_available():
        alloc = torch.cuda.memory_allocated() / (1024**3)
        total = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        logger.info(f"[VRAM - {tag}] {torch.cuda.get_device_name(0)}: {alloc:.2f} GB / {total:.2f} GB allocated")
    else:
        logger.info(f"[VRAM - {tag}] Running on CPU")

# Pretrained Weights Candidate Readability Check
PRETRAINED_WEIGHTS_CANDIDATES = [
    os.path.expanduser("~/rf-detr-base-coco.pth"),
    "/home/jupyter/rf-detr-base-coco.pth",
    "rf-detr-base-coco.pth"
]

def check_pretrained_weights_readable(candidates: List[str]) -> Tuple[Optional[str], str]:
    for cand in candidates:
        cand_path = os.path.expanduser(cand)
        if os.path.exists(cand_path) and os.access(cand_path, os.R_OK) and os.path.getsize(cand_path) > 1024*1024:
            try:
                ckpt = torch.load(cand_path, map_location="cpu", weights_only=False)
                count = len(ckpt.get("model", ckpt).keys()) if isinstance(ckpt, dict) else "?"
                del ckpt
                return cand_path, f"Readable ({os.path.getsize(cand_path)/1e6:.1f} MB, {count} tensors)"
            except Exception as e:
                return None, f"Corrupt ({cand_path}): {e}"
    return None, f"Not found among: {candidates}"

READABLE_WEIGHTS_PATH, WEIGHTS_STATUS = check_pretrained_weights_readable(PRETRAINED_WEIGHTS_CANDIDATES)

logger.info("=" * 65)
logger.info(f"[CELL 1] Core Libraries & Logger Ready (PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()})")
if READABLE_WEIGHTS_PATH:
    logger.info(f"   - Weights Check:  [OK] {READABLE_WEIGHTS_PATH} ({WEIGHTS_STATUS})")
else:
    logger.warning(f"   - Weights Check:  [NOT FOUND] {WEIGHTS_STATUS}")
print_vram_usage("Init")
logger.info("=" * 65)

In [ ]:
# CELL 2 - Model Resolution, Triton Production Bindings & Export Paths
REPO_ROOT = next((p.resolve() for p in [Path("."), Path(".."), Path("../..")] if (p / "coco_files").exists() or (p / ".git").exists()), Path(".").resolve())
PIPELINE_NAME = "multi_class_rfdetr"
PIPELINE_DIR = REPO_ROOT / PIPELINE_NAME
INPUT_JSON_DIR = Path("coco_files") if Path("coco_files").exists() else (REPO_ROOT / "coco_files")
MODEL_VERSION = "v2"  # Direct configuration: no environment override
VERSION_TAG = MODEL_VERSION if MODEL_VERSION else "v1"
VERSION_DIR = PIPELINE_DIR / VERSION_TAG
# Dynamic Model Weight Discovery: Automatically selects the latest or best trained model checkpoint
# 1. Look across all standard version, model, and checkpoint output directories
candidate_model_dirs = [
    VERSION_DIR / "model",
    VERSION_DIR / "checkpoints",
    PIPELINE_DIR / "model",
    PIPELINE_DIR / "checkpoints",
    REPO_ROOT / PIPELINE_NAME / "model",
    REPO_ROOT / PIPELINE_NAME / "checkpoints",
    REPO_ROOT / "model",
    REPO_ROOT / "checkpoints",
    REPO_ROOT / "multi_class_train_rfdetr" / "model",
    REPO_ROOT / "multi_class_train_rfdetr" / "checkpoints",
    Path("/home/jupyter/location_tag/multi_class_rfdetr/v2/checkpoints"),
    Path("/home/jupyter/location_tag/multi_class_rfdetr/v2/model"),
    Path("/home/jupyter/multi_class_rfdetr/v2/checkpoints"),
    Path("/home/jupyter/multi_class_rfdetr/v2/model"),
    Path("model"),
    Path("checkpoints")
]

# 2. Gather all .pth checkpoints across all valid directories
all_discovered_pths = []
for d in candidate_model_dirs:
    if d.exists():
        for p in d.glob("*.pth"):
            if p.is_file() and p.stat().st_size > 1024 * 1024:
                all_discovered_pths.append(p)

# 3. Best Model from Training Discovery & Strict Verification
# RULE: Step 09 ONNX conversion MUST strictly export only the best trained checkpoint: 'best_model.pth'
BEST_MODEL_PATH = None

# Prioritized locations strictly for 'best_model.pth'
strict_best_model_locations = [
    VERSION_DIR / "model" / "best_model.pth",
    VERSION_DIR / "checkpoints" / "best_model.pth",
    PIPELINE_DIR / "model" / "best_model.pth",
    PIPELINE_DIR / "checkpoints" / "best_model.pth",
    REPO_ROOT / PIPELINE_NAME / "model" / "best_model.pth",
    REPO_ROOT / PIPELINE_NAME / "checkpoints" / "best_model.pth",
    REPO_ROOT / "multi_class_train_rfdetr" / "model" / "best_model.pth",
    REPO_ROOT / "multi_class_train_rfdetr" / "checkpoints" / "best_model.pth",
    Path("/home/jupyter/location_tag/multi_class_rfdetr/v2/model/best_model.pth"),
    Path("/home/jupyter/location_tag/multi_class_rfdetr/v2/checkpoints/best_model.pth"),
    Path("/home/jupyter/multi_class_rfdetr/v2/model/best_model.pth"),
    Path("/home/jupyter/multi_class_rfdetr/v2/checkpoints/best_model.pth"),
    REPO_ROOT / "model" / "best_model.pth",
    REPO_ROOT / "best_model.pth",
    Path("model/best_model.pth"),
    Path("checkpoints/best_model.pth")
]

for cand in strict_best_model_locations:
    if cand.exists() and cand.is_file() and cand.stat().st_size > 1024 * 1024:
        BEST_MODEL_PATH = cand
        break

# Secondary search: any discovered .pth strictly named 'best_model.pth'
if BEST_MODEL_PATH is None and all_discovered_pths:
    exact_best = [p for p in all_discovered_pths if p.name == "best_model.pth"]
    if exact_best:
        exact_best.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        BEST_MODEL_PATH = exact_best[0]

# Assertion check: enforce that only best_model.pth is selected
if BEST_MODEL_PATH is None:
    raise FileNotFoundError(
        "CRITICAL ERROR: 'best_model.pth' not found! Notebook 09 requires 'best_model.pth' from training. "
        "Please complete training or verify that 'best_model.pth' exists in the model/checkpoints directory."
    )
elif BEST_MODEL_PATH.name != "best_model.pth":
    raise ValueError(
        f"CRITICAL ERROR: Selected model '{BEST_MODEL_PATH}' is not 'best_model.pth'! "
        "Notebook 09 must always export strictly 'best_model.pth'."
    )

logger.info(f"✓ Confirmed strictly using 'best_model.pth': {BEST_MODEL_PATH} ({BEST_MODEL_PATH.stat().st_size / 1e6:.1f} MB)")

FINAL_MODEL_DIR = BEST_MODEL_PATH.parent if BEST_MODEL_PATH else candidate_model_dirs[0]

EXPORT_DIR = Path("deployment") / "models"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
ONNX_EXPORT_PATH = EXPORT_DIR / "rfdetr_model.onnx"

# Triton tagdet_rt Production Engine Configuration
TRITON_MODEL_NAME = "tagdet_rt"
TRITON_INPUT_NAME = "images"
TRITON_OUTPUT_NAME = "output"
INPUT_HEIGHT = 640
INPUT_WIDTH = 480
TOTAL_OUTPUT_ROWS = 18900
OUTPUT_COLS = 7  # 4 bbox coordinates (cx, cy, w, h) + 3 class scores

# Internal RF-DETR Backbone resolution (must be divisible by backbone patch size, e.g. 560 or 504)
INTERNAL_RESOLUTION = 560

logger.info(f"Target Triton Model:   {TRITON_MODEL_NAME}")
logger.info(f"Source PyTorch Model:  {BEST_MODEL_PATH}")
logger.info(f"Target ONNX Model:     {ONNX_EXPORT_PATH}")
logger.info(f"Input Binding Shape:   ({TRITON_INPUT_NAME}): [1, 3, {INPUT_HEIGHT}, {INPUT_WIDTH}]")
logger.info(f"Output Binding Shape:  ({TRITON_OUTPUT_NAME}): [1, {TOTAL_OUTPUT_ROWS}, {OUTPUT_COLS}]")

In [ ]:
# CELL 14 - Safe Pretrained Weights Loading & Detection Head Patch
from rfdetr.models.lwdetr import LWDETR

# 1. Patch LWDETR.load_state_dict using torch.nn.Module.load_state_dict directly (prevents RecursionError)
def _safe_lwdetr_load_state_dict(self, state_dict, strict=True):
    """Filters out classification heads (class_embed, enc_out_class_embed) from checkpoint when num_classes differs."""
    model_state = self.state_dict()
    filtered_state_dict = {}
    mismatched = []
    
    for k, v in state_dict.items():
        clean_k = k
        if clean_k not in model_state and clean_k.startswith("model."):
            clean_k = clean_k[6:]
        if clean_k not in model_state and clean_k.startswith("module."):
            clean_k = clean_k[7:]
            
        if clean_k in model_state:
            if model_state[clean_k].shape == v.shape:
                filtered_state_dict[clean_k] = v
            else:
                mismatched.append((clean_k, tuple(v.shape), tuple(model_state[clean_k].shape)))
        elif k in model_state:
            if model_state[k].shape == v.shape:
                filtered_state_dict[k] = v
            else:
                mismatched.append((k, tuple(v.shape), tuple(model_state[k].shape)))

    if mismatched:
        logger.info(f"Notice: Filtered {len(mismatched)} classification head layers with class-count mismatch from COCO weights:")
        for name, ckpt_shape, model_shape in mismatched[:4]:
            logger.info(f"   - {name}: checkpoint {ckpt_shape} -> target model {model_shape}")
        if len(mismatched) > 4:
            logger.info(f"   - ... and {len(mismatched) - 4} more classification head tensors.")
        logger.info("Pretrained backbone, transformer, and regression heads loaded successfully.")
    
    # Call base PyTorch nn.Module.load_state_dict directly to avoid any recursion on re-runs
    return torch.nn.Module.load_state_dict(self, filtered_state_dict, strict=False)

LWDETR.load_state_dict = _safe_lwdetr_load_state_dict
logger.info("LWDETR.load_state_dict patched for safe custom num_classes loading.")

# 2. Patch reinitialize_detection_head to handle both class_embed and transformer.enc_out_class_embed
def _fixed_reinitialize(self, num_classes):
    lwdetr = self.model  # the actual LWDETR nn.Module
    device = next(lwdetr.parameters()).device
    
    # Reinitialize class_embed
    if hasattr(lwdetr, "class_embed"):
        in_feat = lwdetr.class_embed.in_features
        lwdetr.class_embed = nn.Linear(in_feat, num_classes).to(device)
        nn.init.normal_(lwdetr.class_embed.weight, std=0.01)
        nn.init.zeros_(lwdetr.class_embed.bias)
        
    # Reinitialize transformer.enc_out_class_embed
    if hasattr(lwdetr, "transformer") and hasattr(lwdetr.transformer, "enc_out_class_embed"):
        enc_heads = lwdetr.transformer.enc_out_class_embed
        if isinstance(enc_heads, nn.ModuleList):
            for i in range(len(enc_heads)):
                in_feat = enc_heads[i].in_features
                enc_heads[i] = nn.Linear(in_feat, num_classes).to(device)
                nn.init.normal_(enc_heads[i].weight, std=0.01)
                nn.init.zeros_(enc_heads[i].bias)
        elif isinstance(enc_heads, nn.Linear):
            in_feat = enc_heads.in_features
            lwdetr.transformer.enc_out_class_embed = nn.Linear(in_feat, num_classes).to(device)
            nn.init.normal_(lwdetr.transformer.enc_out_class_embed.weight, std=0.01)
            nn.init.zeros_(lwdetr.transformer.enc_out_class_embed.bias)
            
    logger.info(f"Detection heads (class_embed + enc_out_class_embed) initialized for num_classes={num_classes}")

rfdetr_main.Model.reinitialize_detection_head = _fixed_reinitialize
logger.info("reinitialize_detection_head patched.")

# 3. Model Instantiation
MODEL_SIZE = globals().get("MODEL_SIZE", "base")  # Direct architecture configuration
RESOLUTION = globals().get("RESOLUTION", globals().get("INTERNAL_RESOLUTION", 560))

if "class_names" not in globals() or not globals().get("class_names"):
    resolved_ann = globals().get("TEST_ANN", None)
    if resolved_ann and os.path.exists(str(resolved_ann)):
        try:
            with open(resolved_ann, "r", encoding="utf-8") as f:
                ann_data = json.load(f)
            class_names = [c["name"] for c in sorted(ann_data.get("categories", []), key=lambda x: x["id"])]
        except Exception:
            class_names = ["blue_aisle", "blue_bay", "location_tag"]
    else:
        class_names = ["blue_aisle", "blue_bay", "location_tag"]
else:
    class_names = globals().get("class_names")

NUM_CLASSES = len(class_names)

model_cls_map = {
    "nano": RFDETRNano,
    "small": RFDETRSmall,
    "medium": RFDETRMedium,
    "base": RFDETRBase,
    "large": RFDETRLarge
}
ModelClass = model_cls_map.get(str(MODEL_SIZE).lower(), RFDETRBase)
model = ModelClass(
    num_classes=NUM_CLASSES,
    pretrain_weights=None,
    resolution=RESOLUTION
)
logger.info(f"RF-DETR-{str(MODEL_SIZE).capitalize()} instantiated with resolution={RESOLUTION}, num_classes={NUM_CLASSES}")

if hasattr(model, "model") and hasattr(model.model, "reinitialize_detection_head"):
    model.model.reinitialize_detection_head(NUM_CLASSES)

# Proactively flush any residual CUDA VRAM allocations from preceding notebook steps
import gc
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Load best_model.pth weights strictly into the initialized architecture on CPU
target_weights = globals().get("BEST_MODEL_PATH", None)
assert target_weights and os.path.exists(str(target_weights)), f"Cannot proceed: best_model.pth missing at {target_weights}"
assert Path(target_weights).name == "best_model.pth", f"Invalid model file: {target_weights}. Must strictly be 'best_model.pth'!"

ckpt_data = torch.load(str(target_weights), map_location="cpu", weights_only=False)
state_dict = ckpt_data.get("model_state_dict", ckpt_data)
model.model.model.load_state_dict(state_dict, strict=False)

epoch_info = f" (Epoch {ckpt_data['epoch']})" if isinstance(ckpt_data, dict) and 'epoch' in ckpt_data else ""
loss_info = f" | Best Val Loss: {ckpt_data.get('best_loss', ckpt_data.get('val_loss', 'N/A'))}" if isinstance(ckpt_data, dict) and ('best_loss' in ckpt_data or 'val_loss' in ckpt_data) else ""
logger.info(f"✓ Strictly verified and loaded weights from best_model.pth: {target_weights}{epoch_info}{loss_info}")

del ckpt_data, state_dict
gc.collect()

# Extract underlying model module on CPU (Zero GPU VRAM usage during ONNX conversion)
device = torch.device("cpu")
lwdetr = model.model.model if hasattr(model, "model") and hasattr(model.model, "model") else getattr(model, "model", model)
lwdetr.to(device)
lwdetr.eval()

# Delete outer model wrapper
del model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

logger.info("✓ Model initialized on CPU for zero-VRAM ONNX graph compilation.")

In [ ]:
# CELL 4 - TagDetRT Production Wrapper & ONNX Graph Generation
import torch.nn.functional as F

class TagDetRTProductionWrapper(torch.nn.Module):
    """
    Drop-in production wrapper for AKS Triton tagdet_rt configuration:
    - Input:  images [1, 3, 640, 480]
    - Resizes internally via bilinear interpolation to (INTERNAL_RESOLUTION, INTERNAL_RESOLUTION)
      to satisfy DINOv2 / transformer backbone divisibility.
    - Runs RF-DETR core model to generate 300 queries (boxes: [cx, cy, w, h], scores: sigmoid(logits)).
    - Concatenates [cx, cy, w, h, s_0, s_1, s_2] -> [1, 300, 7].
    - Zero-pads rows from 300 to 18900 to strictly match Triton output binding dims [18900, 7].
    """
    def __init__(self, core_model, internal_size: int = 560, total_rows: int = 18900, num_classes: int = 3):
        super().__init__()
        self.core = core_model
        self.internal_size = internal_size
        self.total_rows = total_rows
        self.num_classes = num_classes
        
    def forward(self, images: torch.Tensor) -> torch.Tensor:
        # 1. Resize from [B, 3, 640, 480] to backbone resolution [B, 3, internal_size, internal_size]
        x = F.interpolate(images, size=(self.internal_size, self.internal_size), mode="bilinear", align_corners=False)
        
        # 2. Forward pass through RF-DETR
        outputs = self.core(x)
        logits = outputs["pred_logits"]  # [B, 300, num_classes]
        boxes = outputs["pred_boxes"]    # [B, 300, 4] normalized cxcywh
        scores = torch.sigmoid(logits)   # [B, 300, num_classes]
        
        # Slice or pad scores to guarantee exactly num_classes
        if scores.shape[-1] < self.num_classes:
            pad_cls = torch.zeros(scores.shape[0], scores.shape[1], self.num_classes - scores.shape[-1], device=scores.device, dtype=scores.dtype)
            scores = torch.cat([scores, pad_cls], dim=-1)
        elif scores.shape[-1] > self.num_classes:
            scores = scores[..., :self.num_classes]
            
        # 3. Concatenate bbox coords [cx, cy, w, h] + class scores [s_0, s_1, s_2] -> [B, 300, 7]
        det_300 = torch.cat([boxes, scores], dim=-1)
        
        # 4. Zero-pad rows up to total_rows (e.g. 18900)
        pad_rows = self.total_rows - det_300.shape[1]
        if pad_rows > 0:
            zeros = torch.zeros(det_300.shape[0], pad_rows, det_300.shape[2], device=det_300.device, dtype=det_300.dtype)
            output = torch.cat([det_300, zeros], dim=1)  # [B, 18900, 7]
        else:
            output = det_300
            
        return output

# CPU Export Pipeline: Prevents any CUDA Out-of-Memory errors during ONNX symbolic graph expansion
device = torch.device("cpu")
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

wrapper = TagDetRTProductionWrapper(
    lwdetr,
    internal_size=INTERNAL_RESOLUTION,
    total_rows=TOTAL_OUTPUT_ROWS,
    num_classes=OUTPUT_COLS - 4
)
wrapper.to(device)
wrapper.eval()

dummy_input = torch.randn(1, 3, INPUT_HEIGHT, INPUT_WIDTH, device=device)

logger.info(f"Exporting ONNX model strictly conforming to tagdet_rt config:")
logger.info(f" • Input name:  ['{TRITON_INPUT_NAME}'] -> shape: {list(dummy_input.shape)}")
logger.info(f" • Output name: ['{TRITON_OUTPUT_NAME}'] -> shape: [1, {TOTAL_OUTPUT_ROWS}, {OUTPUT_COLS}]")
logger.info(f" • Destination: {ONNX_EXPORT_PATH}")

# =========================================================================
# ONNX Export Compatibility Patch for 'aten::_upsample_bicubic2d_aa'
# In PyTorch, DINOv2 / Vision Transformers use bicubic interpolation with anti-aliasing (aa=True).
# The PyTorch ONNX exporter lacks a native symbolic for aten::_upsample_bicubic2d_aa in Opset 17.
# We apply a dual patch:
# 1. Context manager that transparently disables antialias during ONNX graph tracing.
# 2. Custom ONNX symbolic registration mapping aten::_upsample_bicubic2d_aa to onnx::Resize (cubic mode).
# =========================================================================
from contextlib import contextmanager
from torch.onnx import register_custom_op_symbolic

def _symbolic_upsample_bicubic2d_aa(g, input, output_size, align_corners, scales_h_w=None):
    empty_scales = g.op("Constant", value_t=torch.tensor([], dtype=torch.float32))
    empty_roi = g.op("Constant", value_t=torch.tensor([], dtype=torch.float32))
    align_corners_val = bool(align_corners) if isinstance(align_corners, (bool, int)) else False
    coord_mode = "align_corners" if align_corners_val else "half_pixel"
    return g.op(
        "Resize",
        input,
        empty_roi,
        empty_scales,
        output_size,
        coordinate_transformation_mode_s=coord_mode,
        mode_s="cubic",
        cubic_coeff_a_f=-0.75
    )

try:
    for opset_v in range(11, 21):
        register_custom_op_symbolic("::_upsample_bicubic2d_aa", _symbolic_upsample_bicubic2d_aa, opset_v)
        register_custom_op_symbolic("aten::_upsample_bicubic2d_aa", _symbolic_upsample_bicubic2d_aa, opset_v)
    logger.info("✓ Registered ONNX symbolic handler for 'aten::_upsample_bicubic2d_aa' -> onnx::Resize(mode='cubic')")
except Exception as e:
    logger.warning(f"Symbolic registration note: {e}")

@contextmanager
def disable_interpolate_antialias():
    orig_interpolate = F.interpolate
    def patched_interpolate(*args, **kwargs):
        if "antialias" in kwargs:
            kwargs["antialias"] = False
        return orig_interpolate(*args, **kwargs)
    F.interpolate = patched_interpolate
    try:
        yield
    finally:
        F.interpolate = orig_interpolate

TARGET_ORT_VERSION = "1.16.1"
EXPORT_OPSET = 17

try:
    import onnxruntime as ort
    ort_ver = getattr(ort, "__version__", TARGET_ORT_VERSION)
    logger.info(f"Verified ONNX Runtime: {ort_ver} (Target: {TARGET_ORT_VERSION}) -> Opset {EXPORT_OPSET}")
except Exception as e:
    logger.warning(f"ONNX Runtime check notice: {e}")

# Suppress standard PyTorch TracerWarnings for clean console logs
import warnings
warnings.filterwarnings("ignore", category=torch.jit.TracerWarning)
logger.info("Exporting ONNX graph (TracerWarnings suppressed for clean output)...")

# Perform ONNX export under antialias-safe tracing context
with torch.no_grad(), disable_interpolate_antialias():
    torch.onnx.export(
        wrapper,
        dummy_input,
        str(ONNX_EXPORT_PATH),
        input_names=[TRITON_INPUT_NAME],
        output_names=[TRITON_OUTPUT_NAME],
        opset_version=EXPORT_OPSET,
        do_constant_folding=True
    )
logger.info(f"✓ ONNX export complete! Size: {ONNX_EXPORT_PATH.stat().st_size / 1e6:.1f} MB (Opset {EXPORT_OPSET})")

In [ ]:
# CELL 5 - ONNXRuntime Inference Verification strictly testing [1, 18900, 7] with onnxruntime==1.16.1
import onnxruntime as ort

logger.info(f"Running inference verification with ONNX Runtime v{ort.__version__} (strictly configured for 1.16.1)")
session = ort.InferenceSession(str(ONNX_EXPORT_PATH), providers=["CPUExecutionProvider"])
inputs_meta = session.get_inputs()
outputs_meta = session.get_outputs()

logger.info(f"Verified ONNX Model Bindings:")
for inp in inputs_meta:
    logger.info(f" • Input Binding:  '{inp.name}' | Shape: {inp.shape} | Type: {inp.type}")
for out in outputs_meta:
    logger.info(f" • Output Binding: '{out.name}' | Shape: {out.shape} | Type: {out.type}")

# Assert binding names match tagdet_rt config.pbtxt exactly
assert inputs_meta[0].name == TRITON_INPUT_NAME, f"Input name mismatch: {inputs_meta[0].name} != {TRITON_INPUT_NAME}"
assert outputs_meta[0].name == TRITON_OUTPUT_NAME, f"Output name mismatch: {outputs_meta[0].name} != {TRITON_OUTPUT_NAME}"

# Test inference execution with dummy [1, 3, 640, 480]
dummy_np = np.random.randn(1, 3, INPUT_HEIGHT, INPUT_WIDTH).astype(np.float32)

t0 = time.time()
raw_outputs = session.run([TRITON_OUTPUT_NAME], {TRITON_INPUT_NAME: dummy_np})
latency = (time.time() - t0) * 1000
output_tensor = raw_outputs[0]

logger.info(f"✓ ONNXRuntime session verified successfully in {latency:.1f} ms")
logger.info(f"Output shape: {output_tensor.shape}")
assert output_tensor.shape == (1, TOTAL_OUTPUT_ROWS, OUTPUT_COLS), f"Output shape mismatch: {output_tensor.shape}"

# Check contents: top 300 rows are detections, remaining rows are 0 padding
active_rows = output_tensor[0, :300, :]
padded_rows = output_tensor[0, 300:, :]
logger.info(f"Active queries (0..300): box coords [cx,cy,w,h]={active_rows[0, :4]}, scores={active_rows[0, 4:]}")
logger.info(f"Padded rows (300..18900) max value: {np.max(padded_rows)} (strictly zeros)")
assert np.all(padded_rows == 0.0), "Padded rows must be strictly zeros!"
logger.info("✓ Triton tagdet_rt binding compatibility verification PASSED!")


# Generate and save production Triton config.pbtxt for ONNX Runtime (onnxruntime==1.16.1)
triton_config_content = f"""name: "{TRITON_MODEL_NAME}"
backend: "onnxruntime"
max_batch_size: 1

input [
  {{
    name: "{TRITON_INPUT_NAME}"
    data_type: TYPE_FP32
    dims: [ 3, {INPUT_HEIGHT}, {INPUT_WIDTH} ]
  }}
]

output [
  {{
    name: "{TRITON_OUTPUT_NAME}"
    data_type: TYPE_FP32
    dims: [ {TOTAL_OUTPUT_ROWS}, {OUTPUT_COLS} ]
  }}
]

instance_group [
  {{
    count: 1
    kind: KIND_GPU
  }}
]

default_model_filename: "{ONNX_EXPORT_PATH.name}"
"""

triton_config_paths = [
    EXPORT_DIR / "config.pbtxt",
    REPO_ROOT / "notebook" / "deployment" / "triton" / "config.onnx.pbtxt"
]
for cfg_p in triton_config_paths:
    cfg_p.parent.mkdir(parents=True, exist_ok=True)
    with open(cfg_p, "w", encoding="utf-8") as f:
        f.write(triton_config_content)
    logger.info(f"✓ Generated Triton ONNXRuntime config.pbtxt at: {cfg_p}")